In [6]:
import sys
sys.path.insert(0, "../..")

from ingestion.theirstack_client import TheirStackClient
from infra.snowflake_client import SnowflakeLoader

client = TheirStackClient()

In [7]:
job_ids = [
    735075886, 735321929, 735473065, 735308450, 733389679, 733942488,
    732708479, 732486522, 732681402, 732605030, 732887632, 731239725,
    731700763, 729973405, 729978552, 729466073, 729697490, 726312048,
    725056202, 724454118, 723592915, 724964814, 723592941, 724533888,
    724244447, 723564744, 724187171, 724187177, 722593402, 722684005,
    722998644, 720222233, 721617563, 723641108, 718122706, 719386066,
    723931848, 719339861, 719439128, 719584540,
]

In [8]:
batch_1 = job_ids[:25]
batch_2 = job_ids[25:]

paid_jobs = []

for batch in (batch_1, batch_2):
    result = client._paid_fetch(batch, label="Data Scientist")
    paid_jobs.extend(result)
    print(f"Batch of {len(batch)} → got {len(result)} jobs back")

print(f"Total: {len(paid_jobs)} jobs")

Batch of 25 → got 25 jobs back
Batch of 15 → got 15 jobs back
Total: 40 jobs


In [9]:
# eyeball a couple titles before committing to Snowflake
for job in paid_jobs[:5]:
    print(job.get("job_title"), "—", job.get("company_name") or job.get("company_object", {}).get("name"))

Data Scientist — Findigs, Inc.
Data Scientist, NLP (Systematic Trading) — Thurn Partners
Data Scientist — Scale.jobs
Data Scientist — Gartner
Data Scientist I, SCOT-Inbound, Planning Optimization — Amazon


In [10]:
rows = client._to_snowflake_rows(paid_jobs, label="Data Scientist")

with SnowflakeLoader() as loader:
    load_results = loader.load(rows)
    for table, count in load_results.items():
        print(f"{table}: {count} rows inserted")

RAW.THEIRSTACK.SRC_POSTINGS: 40 rows inserted


In [11]:
import os
import requests

api_key = os.getenv("THEIRSTACK_KEY")

base_filters = {
    "job_title_pattern_not": [
        "(?i)senior", "(?i)\\bsr\\b", "(?i)\\bii\\b", "(?i)\\biii\\b", "(?i)\\biv\\b",
        "(?i)lead", "(?i)principal", "(?i)staff", "(?i)manager", "(?i)director",
        "(?i)head of", "(?i)unpaid", "(?i)non.paid", "(?i)volunteer",
    ],
    "job_seniority_or": ["junior", "mid_level"],
    "job_country_code_or": ["US"],
    "job_location_pattern_or": [
        "(?i)^new york,", "(?i)new york city", "(?i)manhattan", "(?i)brooklyn",
    ],
    "posted_at_max_age_days": 30,  # widened, same as the original failing run
    "order_by": [{"field": "discovered_at", "desc": True}],
    "job_title_pattern_or": ["(?i)data scientist"],
}

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {api_key}",
}

# Walk pages one at a time, print full response on every page —
# so if/when it dies, we see exactly what page and what body.
for page in range(0, 8):
    body = {
        **base_filters,
        "blur_company_data": True,
        "include_total_results": page == 0,
        "limit": 25,
        "page": page,
    }

    response = requests.post(
        "https://api.theirstack.com/v1/jobs/search",
        json=body,
        headers=headers,
        timeout=30,
    )

    print(f"--- page {page} ---")
    print("Status:", response.status_code)

    if response.status_code != 200:
        print("Headers:", dict(response.headers))
        print("Body:", response.text)
        break  # stop as soon as we hit the failure

    data = response.json()
    jobs = data.get("data", [])
    print(f"Got {len(jobs)} jobs, running total so far")

    if not jobs or len(jobs) < 25:
        print("Reached last page naturally — sweep would have completed cleanly.")
        break

--- page 0 ---
Status: 200
Got 25 jobs, running total so far
--- page 1 ---
Status: 200
Got 25 jobs, running total so far
--- page 2 ---
Status: 200
Got 25 jobs, running total so far
--- page 3 ---
Status: 200
Got 25 jobs, running total so far
--- page 4 ---
Status: 200
Got 25 jobs, running total so far
--- page 5 ---
Status: 403
Headers: {'Date': 'Sat, 20 Jun 2026 14:05:00 GMT', 'Content-Type': 'application/json', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'rndr-id': '95b1382b-f804-41d2', 'Server': 'cloudflare', 'vary': 'Accept-Encoding', 'x-render-origin-server': 'uvicorn', 'cf-cache-status': 'DYNAMIC', 'Content-Encoding': 'gzip', 'CF-RAY': 'a0eb5368db2f8815-EWR', 'alt-svc': 'h3=":443"; ma=86400'}
Body: {"request_id":38951514,"error":{"code":"E-020","title":"Premium functionality limitation","description":"Your current plan allows to view up to 5 pages of results."}}
